# Notebook Python confusion matrix generation and json free-text-annotation

#### Import Packages

In [1]:
import pandas as pd
import json
import os

#### Standardization of annotations contained in TSV files

In [2]:
def process_annotation_field(field_value):
    """
    Process annotation fields by:
    - converting to lowercase
    - removing excess whitespace
    - splitting on commas
    - returning a clean list of annotations
    """
    if pd.isna(field_value) or field_value == "" or field_value is None:
        return []

    # Convert to lowercase
    text = str(field_value).lower()

    # Normalize excessive whitespace
    text = " ".join(text.split())

    # Split by commas and strip whitespace
    annotations = [item.strip() for item in text.split(",")]

    # Remove empty stringsSingle_Cell_gold_standard_consensus
    annotations = [item for item in annotations if item]

    return annotations

#### Load tsv file, creat json and load json file

In [3]:
def load_tsv_and_create_json(tsv_file, json_file, discipline):
    """
    Load TSV and create JSON including all sources
    """
    df = pd.read_csv(tsv_file, sep="\t", encoding="utf-8")
    df.columns = df.columns.str.strip()

    all_tools = {}
    counter = 1

    for _, row in df.iterrows():
        tool_key = f"tools#{counter}"
        source = row.get("Source", "").strip()

        tool_data = {
            "name": row.get("Tool name", "").strip(),
            "discipline": discipline,
            "source": source,
            "topic": process_annotation_field(row.get("Topic", "")),
            "operation": process_annotation_field(row.get("Operation", "")),
            "input data": process_annotation_field(row.get("Input data", "")),
            "output data": process_annotation_field(row.get("Output data", "")),
            "input format": process_annotation_field(row.get("Input format", "")),
            "output format": process_annotation_field(row.get("Output format", "")),
        }

        all_tools[tool_key] = tool_data
        counter += 1

    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(all_tools, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(all_tools)} tools to {json_file}")


def load_json(json_file):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return None

#### Calculates confusion matrices for the LLM 

In [ ]:
def calculate_metrics_per_tool_annotation(json_data):
    """
    For each tool and each annotation type for LLM files, calculate TP, FP, FN, 
    TN, precision, recall, F1_score, Total annotation retained in consensus 
    expert-LLM, not retained and mixed annotations count.  
    """

    annotation_fields = [
        "topic",
        "operation",
        "input data",
        "output data",
        "input format",
        "output format",
    ]

    results = []

    tool_names = set(entry["name"] for entry in json_data.values())

    for tool in tool_names:
        prov_sets = {
            "consensus expert llm": {field: set() for field in annotation_fields},
            "proposition deepseek-v3.1": {field: set() for field in annotation_fields},
            "consensus expert": {field: set() for field in annotation_fields},
        }

        for entry in json_data.values():
            if entry["name"] != tool:
                continue

            prov = entry["source"].strip().lower()

            if prov not in prov_sets:
                continue

            for field in annotation_fields:
                prov_sets[prov][field].update(entry.get(field, []))

        # Compute metrics
        for field in annotation_fields:
            final_consensus_set = prov_sets["consensus expert llm"][field]
            deepseek_set = prov_sets["proposition deepseek-v3.1"][field]
            expert_set = prov_sets["consensus expert"][field]

            missing_in_deepseek = final_consensus_set - deepseek_set

            FN_set = {x for x in missing_in_deepseek if x in expert_set}
            mixed_tp = {x for x in missing_in_deepseek if x not in expert_set}

            TP_set = (final_consensus_set & deepseek_set) | mixed_tp
            FP_set = deepseek_set - final_consensus_set
            TN_set = set()

            deepseek_total = len(deepseek_set)

            deepseek_retained = deepseek_set & final_consensus_set
            deepseek_not_retained = deepseek_set - final_consensus_set

            mixed_annotations = mixed_tp  # your definition

            # Counts
            deepseek_retained_count = len(deepseek_retained)
            deepseek_not_retained_count = len(deepseek_not_retained)
            mixed_annotations_count = len(mixed_annotations)

            TP = len(TP_set)
            FP = len(FP_set)
            FN = len(FN_set)
            TN = 0

            precision = round(TP / (TP + FP) if (TP + FP) else 0, 2)
            recall = round(TP / (TP + FN) if (TP + FN) else 0, 2)
            f1_score = round(
                (2 * precision * recall / (precision + recall)
                 if (precision + recall) else 0),
                2,
            )

            results.append({
                "tool_name": tool,
                "annotation_type": field,

                # Confusion metrics
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "TN": TN,
                "precision": precision,
                "recall": recall,
                "f1_score": f1_score,

                # DeepSeek stats
                "deepseek_total": deepseek_total,
                "deepseek_retained_in_consensus": deepseek_retained_count,
                "deepseek_not_retained": deepseek_not_retained_count,
                "mixed_annotations_count": mixed_annotations_count,

                # Detailed annotation lists
                "TP_annotations": sorted(TP_set),
                "FP_annotations": sorted(FP_set),
                "FN_annotations": sorted(FN_set),

                "deepseek_retained_annotations": sorted(deepseek_retained),
                "deepseek_not_retained_annotations": sorted(deepseek_not_retained),
                "mixed_annotations": sorted(mixed_annotations),
            })

    return results

#### Calculates confusion matrices for the experts

In [ ]:
def calculate_metrics_expert_contribution(json_data):
    """
    For each tool and each annotation type for expert files, calculate TP, FP, FN, 
    TN, precision, recall, F1_score, Total annotation retained and not retained
    in consensus expert-LLM.  
    """

    annotation_fields = [
        "topic",
        "operation",
        "input data",
        "output data",
        "input format",
        "output format",
    ]

    results = []

    tool_names = set(entry["name"] for entry in json_data.values())

    expected_prov = {
        "consensus expert llm",
        "proposition deepseek-v3.1",
        "consensus expert",
    }

    for tool in tool_names:
        prov_sets = {
            "consensus expert llm": {field: set() for field in annotation_fields},
            "proposition deepseek-v3.1": {field: set() for field in annotation_fields},
            "consensus expert": {field: set() for field in annotation_fields},
        }

        for entry in json_data.values():
            if entry["name"] != tool:
                continue

            prov = entry["source"].strip().lower()
            if prov not in expected_prov:
                continue

            for field in annotation_fields:
                prov_sets[prov][field].update(entry.get(field, []))

        # compute metrics per annotation field
        for field in annotation_fields:
            final_consensus_set = prov_sets["consensus expert llm"][field]
            expert_set = prov_sets["consensus expert"][field]
            deepseek_set = prov_sets["proposition deepseek-v3.1"][field]

            missing_in_expert = final_consensus_set - expert_set

            FN_set = {x for x in missing_in_expert if x not in deepseek_set}
            mixed_tp = {x for x in missing_in_expert if x in deepseek_set}

            TP_set = (final_consensus_set & expert_set) | mixed_tp
            FP_set = expert_set - final_consensus_set
            TN_set = set()

            expert_total = len(expert_set)

            expert_retained = expert_set & final_consensus_set
            expert_not_retained = expert_set - final_consensus_set

            expert_retained_count = len(expert_retained)
            expert_not_retained_count = len(expert_not_retained)

            TP = len(TP_set)
            FP = len(FP_set)
            FN = len(FN_set)
            TN = 0

            precision = round(TP / (TP + FP) if (TP + FP) else 0, 2)
            recall = round(TP / (TP + FN) if (TP + FN) else 0, 2)
            f1_score = round(
                (2 * precision * recall / (precision + recall)
                 if (precision + recall) else 0),
                2,
            )

            results.append({
                "tool_name": tool,
                "annotation_type": field,

                # Confusion metrics
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "TN": TN,
                "precision": precision,
                "recall": recall,
                "f1_score": f1_score,

                # Expert stats
                "expert_total": expert_total,
                "expert_retained_in_consensus": expert_retained_count,
                "expert_not_retained": expert_not_retained_count,

                # Annotation details
                "TP_annotations": sorted(TP_set),
                "FP_annotations": sorted(FP_set),
                "FN_annotations": sorted(FN_set),

                "expert_retained_annotations": sorted(expert_retained),
                "expert_not_retained_annotations": sorted(expert_not_retained),
            })

    return results

#### Pipeline to generate JSON files, calculate and save a confusion matrix associated

In [6]:
def save_metrics_to_tsv(metrics, tsv_file):
    """
    Save metrics dictionary/list to a TSV file.
    """
    df = pd.DataFrame(metrics)
    df.to_csv(tsv_file, sep="\t", index=False)
    print(f"Saved confusion metrics to {tsv_file}")


def run_pipeline_all_domains(
    raw_base="Raw_free_text_annotation",
    output_base="Confusion_matrix_free_text_annotation"
):
    """
    Run full pipeline (LLM + expert) for all domains
    - Read TSV from Raw_free_text_annotation
    - Save JSON in Raw folder
    - Save metrics in Confusion_matrix_free_text_annotation
    """

    domains = [
        "Metagenomic",
        "Systems_biology",
        "Single_Cell",
        "Phylogeny",
        "Neuroimaging",
        "Genetic_variant",
        "Microscopy",
    ]

    for domain in domains:
        print(f"\nProcessing domain: {domain}")

        # INPUT (RAW DATA)
        raw_domain_folder = os.path.join(raw_base, domain)

        tsv_file = os.path.join(
            raw_domain_folder,
            f"{domain}_gold_standard_consensus.tsv"
        )

        json_file = os.path.join(
            raw_domain_folder,
            f"{domain}_tools_annotations.json"
        )

        # OUTPUT (METRICS)
        output_domain_folder = os.path.join(output_base, domain)

        os.makedirs(output_domain_folder, exist_ok=True)

        # STEP 1: TSV to JSON
        load_tsv_and_create_json(
            tsv_file=tsv_file,
            json_file=json_file,
            discipline=domain
        )

        # STEP 2: LOAD JSON
        data = load_json(json_file)
        if not data:
            continue

        # STEP 3: LLM METRICS
        metrics_llm = calculate_metrics_per_tool_annotation(data)

        save_metrics_to_tsv(
            metrics_llm,
            tsv_file=os.path.join(
                output_domain_folder,
                f"{domain}_confusion_metrics_LLM.tsv"
            )
        )

        # STEP 4: EXPERT METRICS
        metrics_expert = calculate_metrics_expert_contribution(data)

        save_metrics_to_tsv(
            metrics_expert,
            tsv_file=os.path.join(
                output_domain_folder,
                f"{domain}_confusion_metrics_expert.tsv"
            )
        )

#### Run all the pipeline

In [7]:
run_pipeline_all_domains()


Processing domain: Metagenomic
Saved 36 tools to Raw_free_text_annotation/Metagenomic/Metagenomic_tools_annotations.json
Saved confusion metrics to Confusion_matrix_free_text_annotation/Metagenomic/Metagenomic_confusion_metrics_LLM.tsv
Saved confusion metrics to Confusion_matrix_free_text_annotation/Metagenomic/Metagenomic_confusion_metrics_expert.tsv

Processing domain: Systems_biology
Saved 37 tools to Raw_free_text_annotation/Systems_biology/Systems_biology_tools_annotations.json
Saved confusion metrics to Confusion_matrix_free_text_annotation/Systems_biology/Systems_biology_confusion_metrics_LLM.tsv
Saved confusion metrics to Confusion_matrix_free_text_annotation/Systems_biology/Systems_biology_confusion_metrics_expert.tsv

Processing domain: Single_Cell
Saved 15 tools to Raw_free_text_annotation/Single_Cell/Single_Cell_tools_annotations.json
Saved confusion metrics to Confusion_matrix_free_text_annotation/Single_Cell/Single_Cell_confusion_metrics_LLM.tsv
Saved confusion metrics t